# MODUL PRAKTIKUM BIG DATA
## Pertemuan 12 — Klasifikasi & ML Pipeline dengan Spark MLlib

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 12 |
| **Topik** | Klasifikasi (Logistic Regression, Random Forest), ML Pipeline, Evaluasi Model Klasifikasi |
| **Estimasi Waktu** | 150 menit |
| **Prasyarat** | Modul Pertemuan 1-11 selesai (memahami Spark MLlib dasar: `StringIndexer`, `VectorAssembler`, regresi) |

---

### Cara Menggunakan Modul Ini

**Tidak ada instalasi baru.** Seluruh modul dikerjakan langsung di Jupyter Notebook, melanjutkan modul `pyspark.ml` yang sudah dikenalkan Pertemuan 11.

> **Konsistensi versi:** Tetap **Spark/PySpark 3.5.9**, **Python 3.11** — tidak ada perubahan versi apa pun.


---
## Recap Pertemuan Sebelumnya

- [ ] Memahami pola `fit()`/`transform()`, `StringIndexer`, `VectorAssembler` dari Pertemuan 11
- [ ] Memahami cara melatih & mengevaluasi model `LinearRegression`

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 12, mahasiswa mampu:
1. Menjelaskan perbedaan tugas klasifikasi dan regresi.
2. Merangkai tahapan feature engineering dan pemodelan menjadi satu **ML Pipeline**.
3. Melatih dan membandingkan dua algoritma klasifikasi: **Logistic Regression** dan **Random Forest**.
4. Mengevaluasi model klasifikasi menggunakan accuracy, F1-score, confusion matrix, dan AUC-ROC.

---

## 12.1 Klasifikasi vs Regresi

Pada Pertemuan 11, anda telah memprediksi **angka kontinu** (pendapatan) — itu adalah tugas **regresi**. Hari ini anda mempelajari **klasifikasi**: memprediksi **kategori/kelas** (mis. "Ya"/"Tidak", "Tinggi"/"Rendah", atau beberapa kelas sekaligus).

| Aspek | Regresi (Pertemuan 11) | Klasifikasi (Pertemuan 12) |
|---|---|---|
| **Target/label yang diprediksi** | Angka kontinu (mis. Rp 850.000) | Kategori/kelas (mis. 0 atau 1) |
| **Contoh kasus** | Memprediksi total pendapatan | Memprediksi transaksi bernilai tinggi/rendah |
| **Algoritma pada modul ini** | `LinearRegression` | `LogisticRegression`, `RandomForestClassifier` |
| **Metrik evaluasi** | RMSE, R² | Accuracy, F1, AUC-ROC |

**Studi kasus hari ini:** memprediksi apakah sebuah transaksi tergolong **"bernilai tinggi" (label 1) atau "bernilai biasa" (label 0)** berdasarkan atributnya — berguna misalnya untuk menentukan transaksi mana yang layak diprioritaskan tim customer service, atau dijadikan target promosi loyalitas pelanggan.


---
## 12.2 Persiapan: SparkSession & Dataset

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan12-MLlibKlasifikasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 14:08:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [5]:
import numpy as np
import pandas as pd

np.random.seed(21)
n = 3000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo"]
metode_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

data = pd.DataFrame({
    "unit_terjual": np.random.randint(1, 15, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "metode_pembayaran": np.random.choice(metode_list, size=n),
})
data["total_pendapatan"] = data["unit_terjual"] * data["harga_satuan"]

# Label: 1 = transaksi "bernilai tinggi", 0 = biasa
# Dibuat probabilistik (bukan aturan kaku) agar mencerminkan pola real world scenario yang tidak sempurna
threshold = data["total_pendapatan"].median()
prob_tinggi = 1 / (1 + np.exp(-(data["total_pendapatan"] - threshold) / 200000))
data["label"] = (np.random.rand(n) < prob_tinggi).astype(int)

df = spark.createDataFrame(data)
df.show(5)
print("\nDistribusi label:")
df.groupBy("label").count().show()

+------------+------------+------------+----------+-----------------+----------------+-----+
|unit_terjual|harga_satuan|    kategori|      kota|metode_pembayaran|total_pendapatan|label|
+------------+------------+------------+----------+-----------------+----------------+-----+
|          10|      150000|     Makanan|  Magelang|     Kartu Kredit|         1500000|    1|
|           9|      100000|   Kesehatan|  Semarang|              COD|          900000|    1|
|           5|       25000|   Kesehatan|  Semarang|         E-Wallet|          125000|    0|
|           1|       50000|     Makanan|Yogyakarta|     Kartu Kredit|           50000|    0|
|           1|      100000|Rumah Tangga|      Solo|         E-Wallet|          100000|    0|
+------------+------------+------------+----------+-----------------+----------------+-----+
only showing top 5 rows


Distribusi label:


[Stage 1:>                                                          (0 + 4) / 4]

+-----+-----+
|label|count|
+-----+-----+
|    1| 1518|
|    0| 1482|
+-----+-----+



Perhatikan distribusi label yang **relatif seimbang** (mendekati 50:50) — ini penting diperhatikan, karena dataset dengan distribusi kelas yang sangat timpang (mis. 99:1) membutuhkan penanganan khusus yang belum kita bahas di modul ini.


---
## 12.3 Mengapa ML Pipeline?

Pada Pertemuan 11, anda menjalankan setiap tahap (indexing, assembling, pelatihan) **satu per satu secara manual**. Cara ini berisiko human error (mis. lupa menerapkan `StringIndexer` yang sama pada data baru — inilah yang secara sengaja diingatkan di instruksi Tugas Mandiri Pertemuan 11!). **`Pipeline`** merangkai seluruh tahap menjadi **satu alur kerja tunggal** yang dapat di-`fit()` dan `transform()` sekali jalan — jauh lebih rapi dan minim kesalahan, terutama saat model akan dipakai berulang kali pada data baru di kemudian hari (produksi).

### 12.3.1 Menyiapkan Tahapan (*Stages*) Pipeline

Kali ini kita juga menambahkan satu komponen baru: **`OneHotEncoder`** — setelah `StringIndexer` mengubah kategori menjadi angka (0, 1, 2, ...), `OneHotEncoder` mengubahnya lagi menjadi representasi biner, agar model tidak salah mengira urutan angka indeks sebagai "urutan besar-kecil" yang sebetulnya tidak ada (kategori "Fashion" bukan berarti "lebih besar" dari kategori "Elektronik" hanya karena indeksnya lebih besar).

In [6]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Tahap 1-3: StringIndexer untuk setiap kolom kategorikal
idx_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
idx_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")
idx_metode = StringIndexer(inputCol="metode_pembayaran", outputCol="metode_idx")

# Tahap 4: OneHotEncoder — mengubah ketiga kolom indeks menjadi representasi biner sekaligus
ohe = OneHotEncoder(
    inputCols=["kategori_idx", "kota_idx", "metode_idx"],
    outputCols=["kategori_vec", "kota_vec", "metode_vec"]
)

# Tahap 5: VectorAssembler — menggabungkan seluruh fitur menjadi satu kolom "features"
assembler = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "kategori_vec", "kota_vec", "metode_vec"],
    outputCol="features"
)

# Tahap 6: Algoritma klasifikasi
lr = LogisticRegression(featuresCol="features", labelCol="label")

print("Keenam tahap pipeline berhasil didefinisikan (belum dijalankan).")

Keenam tahap pipeline berhasil didefinisikan (belum dijalankan).


### 12.3.2 Merangkai dan Menjalankan Pipeline

In [7]:
# Pipeline merangkai SELURUH tahap menjadi satu alur kerja
pipeline = Pipeline(stages=[idx_kategori, idx_kota, idx_metode, ohe, assembler, lr])

# Split data SEBELUM fit — prinsip yang sama seperti Pertemuan 11
train, test = df.randomSplit([0.8, 0.2], seed=42)

# fit() SEKALI SAJA menjalankan seluruh 6 tahap secara berurutan pada data training
model = pipeline.fit(train)

print("Pipeline berhasil dilatih!")
print(f"Data training: {train.count()} | Data testing: {test.count()}")

Pipeline berhasil dilatih!
Data training: 2426 | Data testing: 574


### 12.3.3 Membuat Prediksi

In [8]:
# transform() pada data testing juga otomatis menjalankan SELURUH tahap
# (indexing, encoding, assembling) sebelum akhirnya menghasilkan prediksi -- tidak ada risiko lupa satu tahap
predictions = pipeline_model_columns = model.transform(test)

predictions.select("kategori", "kota", "metode_pembayaran", "label", "prediction", "probability").show(10, truncate=False)

+------------+----------+-----------------+-----+----------+-----------------------------------------+
|kategori    |kota      |metode_pembayaran|label|prediction|probability                              |
+------------+----------+-----------------+-----+----------+-----------------------------------------+
|Elektronik  |Yogyakarta|Transfer Bank    |0    |0.0       |[0.9636674878170381,0.03633251218296185] |
|Makanan     |Yogyakarta|E-Wallet         |0    |0.0       |[0.9745390449411032,0.025460955058896784]|
|Rumah Tangga|Magelang  |Kartu Kredit     |0    |0.0       |[0.9759221630009688,0.02407783699903121] |
|Fashion     |Magelang  |COD              |0    |0.0       |[0.9643478504366934,0.035652149563306645]|
|Elektronik  |Solo      |COD              |0    |0.0       |[0.9192365383434341,0.0807634616565659]  |
|Fashion     |Semarang  |E-Wallet         |0    |0.0       |[0.9135853015468247,0.08641469845317529] |
|Kesehatan   |Yogyakarta|COD              |0    |0.0       |[0.9255852017

Kolom **`probability`** menunjukkan tingkat keyakinan model terhadap masing-masing kelas (`[probabilitas kelas 0, probabilitas kelas 1]`), sementara **`prediction`** adalah kelas dengan probabilitas tertinggi.

---

## 12.4 Evaluasi Model Klasifikasi

In [9]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Accuracy -- persentase prediksi yang benar secara keseluruhan
ev_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = ev_acc.evaluate(predictions)

# F1-score -- rata-rata harmonik precision & recall, lebih andal dibanding accuracy saat kelas tidak seimbang
ev_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1 = ev_f1.evaluate(predictions)

# AUC-ROC -- mengukur kemampuan model membedakan kedua kelas di berbagai ambang batas (threshold)
ev_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = ev_auc.evaluate(predictions)

print(f"Accuracy : {accuracy:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"AUC-ROC  : {auc:.4f}")

Accuracy : 0.7892
F1-score : 0.7894
AUC-ROC  : 0.8802


### Confusion Matrix

*Confusion matrix* menunjukkan **rincian** jenis kesalahan model — apakah model lebih sering keliru memprediksi kelas 0 sebagai 1, atau sebaliknya. Kita dapat membuatnya sederhana menggunakan `groupBy` yang sudah sangat kita kuasai sejak Pertemuan 4.

In [10]:
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

# Interpretasi:
# label=0, prediction=0.0 -> True Negative  (benar memprediksi kelas 0)
# label=0, prediction=1.0 -> False Positive (salah, seharusnya 0 tapi diprediksi 1)
# label=1, prediction=0.0 -> False Negative (salah, seharusnya 1 tapi diprediksi 0)
# label=1, prediction=1.0 -> True Positive  (benar memprediksi kelas 1)

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|  240|
|    0|       1.0|   66|
|    1|       0.0|   55|
|    1|       1.0|  213|
+-----+----------+-----+



| Metrik | Interpretasi |
|---|---|
| **Accuracy** | Proporsi prediksi benar dari total. Bisa menyesatkan jika kelas sangat timpang. |
| **F1-score** | Menyeimbangkan *precision* (dari semua yang diprediksi positif, berapa yang benar) dan *recall* (dari semua yang sebenarnya positif, berapa yang berhasil ditemukan). |
| **AUC-ROC** | Skala 0.5 (setara tebak acak) hingga 1.0 (sempurna). Mengukur kemampuan model memisahkan kedua kelas secara umum. |


---
## 12.5 Membandingkan Algoritma: Logistic Regression vs Random Forest

Salah satu keunggulan menggunakan `Pipeline` adalah **kemudahan mengganti algoritma** — kita cukup mengganti tahap terakhir saja, seluruh tahap feature engineering (indexing, encoding, assembling) tetap dipakai ulang tanpa perubahan.

In [11]:
from pyspark.ml.classification import RandomForestClassifier

# Mendefinisikan algoritma BERBEDA -- Random Forest, sebuah ensemble dari banyak decision tree
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20)

# Pipeline BARU -- lima tahap awal identik, hanya tahap terakhir yang diganti
pipeline_rf = Pipeline(stages=[idx_kategori, idx_kota, idx_metode, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
predictions_rf = model_rf.transform(test)

accuracy_rf = ev_acc.evaluate(predictions_rf)
auc_rf = ev_auc.evaluate(predictions_rf)

print("=== Perbandingan Hasil ===")
print(f"Logistic Regression -> Accuracy: {accuracy:.4f} | AUC: {auc:.4f}")
print(f"Random Forest        -> Accuracy: {accuracy_rf:.4f} | AUC: {auc_rf:.4f}")

=== Perbandingan Hasil ===
Logistic Regression -> Accuracy: 0.7892 | AUC: 0.8802
Random Forest        -> Accuracy: 0.8171 | AUC: 0.8896


> **Tidak selalu model yang lebih kompleks (Random Forest) mengungguli model yang lebih sederhana (Logistic Regression)** — performa sangat bergantung pada karakteristik data. Pada praktik nyata, membandingkan beberapa algoritma seperti ini adalah langkah standar sebelum memutuskan model mana yang akan dipakai. Jangan berasumsi "yang lebih canggih pasti lebih baik" tanpa benar-benar mengujinya.

---

## Menutup SparkSession

In [16]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dan dataset (Sub-bab 12.2) sebelum mengerjakan latihan berikut.

In [12]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan12").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.createDataFrame(data)
train, test = df.randomSplit([0.8, 0.2], seed=42)
print("Siap untuk latihan.")

Siap untuk latihan.


**Soal 1.** Buat `Pipeline` baru menggunakan algoritma `DecisionTreeClassifier` (impor dari `pyspark.ml.classification`) sebagai tahap terakhir, dengan lima tahap feature engineering yang identik dengan Sub-bab 12.3. Latih pipeline tersebut pada data `train`.

In [13]:
# Jawaban Soal 1 di sini
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml import Pipeline

# Membuat algoritma Decision Tree
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label"
)

# Membuat Pipeline dengan lima tahap feature engineering
pipeline_dt = Pipeline(stages=[
    idx_kategori,
    idx_kota,
    idx_metode,
    ohe,
    assembler,
    dt
])

# Melatih model menggunakan data training
model_dt = pipeline_dt.fit(train)

print("Pipeline Decision Tree berhasil dilatih!")

Pipeline Decision Tree berhasil dilatih!


**Soal 2.** Evaluasi model dari Soal 1 pada data `test` menggunakan **accuracy** dan **AUC-ROC**. Bandingkan hasilnya dengan Logistic Regression dan Random Forest pada Sub-bab 12.5 — model mana yang terbaik dari ketiganya?

In [14]:
# Jawaban Soal 2 di sini
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)

# Membuat prediksi Decision Tree
predictions_dt = model_dt.transform(test)

# Evaluator Accuracy
ev_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# Evaluator AUC-ROC
ev_auc = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

# Evaluasi Logistic Regression
accuracy = ev_acc.evaluate(model.transform(test))
auc = ev_auc.evaluate(model.transform(test))

# Evaluasi Random Forest
predictions_rf = model_rf.transform(test)
accuracy_rf = ev_acc.evaluate(predictions_rf)
auc_rf = ev_auc.evaluate(predictions_rf)

# Evaluasi Decision Tree
accuracy_dt = ev_acc.evaluate(predictions_dt)
auc_dt = ev_auc.evaluate(predictions_dt)

# Menampilkan perbandingan
print("=== PERBANDINGAN HASIL MODEL ===")
print(f"Logistic Regression -> Accuracy: {accuracy:.4f} | AUC: {auc:.4f}")
print(f"Random Forest        -> Accuracy: {accuracy_rf:.4f} | AUC: {auc_rf:.4f}")
print(f"Decision Tree        -> Accuracy: {accuracy_dt:.4f} | AUC: {auc_dt:.4f}")

=== PERBANDINGAN HASIL MODEL ===
Logistic Regression -> Accuracy: 0.7892 | AUC: 0.8802
Random Forest        -> Accuracy: 0.8171 | AUC: 0.8896
Decision Tree        -> Accuracy: 0.8380 | AUC: 0.8330


**Soal 3.** Menggunakan `predictions` dari model Logistic Regression (Sub-bab 12.4), hitung **secara manual** nilai *precision* untuk kelas 1 (petunjuk: precision = True Positive / (True Positive + False Positive), keempat angka tersebut dapat dihitung dari hasil `groupBy("label","prediction").count()` yang sudah dibuat sebelumnya).

In [15]:
from pyspark.sql.functions import col

# Membuat kembali prediksi Logistic Regression
predictions = model.transform(test)

# Menghitung jumlah data berdasarkan label dan prediction
confusion = predictions.groupBy("label", "prediction").count()

# Menampilkan hasil perhitungan
hasil = confusion.collect()

print("=== CONFUSION MATRIX ===")
for row in hasil:
    print(row)

# Menghitung True Positive (TP)
TP = predictions.filter(
    (col("label") == 1) & (col("prediction") == 1)
).count()

# Menghitung False Positive (FP)
FP = predictions.filter(
    (col("label") == 0) & (col("prediction") == 1)
).count()

# Menghitung Precision
precision = TP / (TP + FP) if (TP + FP) > 0 else 0

print("\n=== HASIL PRECISION ===")
print("True Positive :", TP)
print("False Positive:", FP)
print("Precision     :", round(precision, 4))

=== CONFUSION MATRIX ===
Row(label=1, prediction=1.0, count=213)
Row(label=0, prediction=1.0, count=66)
Row(label=0, prediction=0.0, count=240)
Row(label=1, prediction=0.0, count=55)

=== HASIL PRECISION ===
True Positive : 213
False Positive: 66
Precision     : 0.7634


**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: mengapa `Pipeline` sangat membantu ketika kita ingin membandingkan beberapa algoritma sekaligus (seperti Sub-bab 12.5), dibandingkan menjalankan setiap tahap secara manual satu per satu seperti pada Pertemuan 11? Tulis jawaban pada markdown cell di bawah ini.
Pipeline sangat membantu karena seluruh tahapan pengolahan data hingga pelatihan model dapat digabungkan menjadi satu proses. Dengan begitu, kita dapat membandingkan beberapa algoritma dengan lebih mudah tanpa harus mengulang setiap tahapan secara manual. Selain itu, Pipeline juga dapat mengurangi kesalahan saat melakukan pengolahan data dan prediksi.


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 13 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

"Kopi Nusantara" ingin membangun model untuk memprediksi apakah seorang pelanggan akan memberikan **ulasan positif (rating ≥ 4) atau tidak**, berdasarkan atribut pesanan mereka — berguna untuk mengidentifikasi kombinasi menu/cabang yang berisiko mengecewakan pelanggan sebelum benar-benar terjadi.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

### Instruksi Pengerjaan

Buat notebook baru **`Tugas12_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas12_ulasan.csv`, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Membangun ML Pipeline** *(bobot 25%)*

Buat `Pipeline` dengan tahapan: `StringIndexer` untuk `nama_menu`, `cabang`, `waktu_pesan` → `OneHotEncoder` untuk ketiganya → `VectorAssembler` (gabungkan hasil one-hot encoding dengan `qty` dan `waktu_tunggu_menit`) → `LogisticRegression` dengan `labelCol="ulasan_positif"`.

**B. Pelatihan & Evaluasi** *(bobot 25%)*

Bagi data 80:20 (`seed=77`), latih pipeline, lalu evaluasi pada data testing menggunakan **accuracy**, **F1-score**, dan **AUC-ROC**. Tampilkan juga *confusion matrix* sederhana menggunakan `groupBy`.

**C. Model Pembanding** *(bobot 20%)*

Latih **`RandomForestClassifier`** (`numTrees=30`) menggunakan pipeline dengan lima tahap feature engineering yang identik dari bagian A (hanya tahap terakhir yang diganti). Bandingkan ketiga metrik evaluasinya dengan model Logistic Regression pada bagian B.

**D. Analisis Kesalahan Model** *(bobot 15%)*

Dari hasil prediksi model **terbaik** (pilih salah satu dari B atau C berdasarkan AUC tertinggi), tampilkan 10 baris di mana model **salah** memprediksi (`label != prediction`). Amati: apakah ada pola tertentu pada kesalahan tersebut (mis. mayoritas terjadi pada `waktu_tunggu_menit` yang berada di sekitar ambang batas tertentu)?

**E. Kesimpulan Bisnis** *(bobot 15%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan model yang telah dibangun, apa rekomendasi konkret yang dapat diberikan kepada manajemen "Kopi Nusantara" untuk meningkatkan kepuasan pelanggan? Kaitkan dengan fitur yang paling berpengaruh terhadap prediksi (boleh merujuk pada pola yang ditemukan di bagian D).

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas12_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Seluruh cell wajib sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. ML Pipeline | Seluruh tahap pipeline terdefinisi dengan benar & berurutan logis | 25% |
| B. Pelatihan & Evaluasi | Split, pelatihan, dan ketiga metrik evaluasi benar | 25% |
| C. Model Pembanding | Random Forest terlatih benar & perbandingan dijelaskan | 20% |
| D. Analisis Kesalahan | Baris kesalahan ditampilkan & pola dianalisis dengan masuk akal | 15% |
| E. Kesimpulan Bisnis | Rekomendasi konkret, relevan, dan berdasarkan temuan model | 15% |
